# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant
# Also install matplotlib for visualization if not present
!pip install matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
# Print summary
print("%s\n\n%s" % (metadata.name, metadata.description))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their @id
record_sets = list(dataset.record_sets)
print("Record Sets:")
for rset in record_sets:
    print(f"  @id: {rset['@id']} | name: {rset.get('name', '<no name>')} | description: {rset.get('description', '<no description>')}")

# List fields for each record set with their @id
for rset in record_sets:
    print(f"\nFields for record set '@id': {rset['@id']}, name: {rset.get('name', '<no name>')}")
    for field in rset.get('field', []):
        print(f"    Field @id: {field['@id']}, name: {field.get('name', '<no name>')}, dataType: {field.get('dataType', '<no dataType>')}, column @id: {field.get('column', {}).get('@id', '<no column>')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set
record_set_ids = [rset['@id'] for rset in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display columns for the first record set
if record_set_ids:
    first_record_set_id = record_set_ids[0]
    print(f"Columns for record set @id '{first_record_set_id}':")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalizing numeric fields, and grouping data by key attributes.

In [ ]:
# Example EDA: Filter and normalize a numeric field from Table 2 (e.g., hormone levels)
# Identify record set @id and a numeric field @id from overview

# For demonstration, let's select the second record set (Table 2) as an example
if len(record_set_ids) > 1:
    table2_id = record_set_ids[1]
    df = dataframes[table2_id]

    # Find numeric fields
    record_set = next(rset for rset in dataset.record_sets if rset['@id'] == table2_id)
    numeric_fields = [f for f in record_set.get('field', []) if f.get('dataType', '').lower() in ['float', 'integer', 'number']]

    if numeric_fields:
        numeric_field_id = numeric_fields[0]['@id']
        # Attempt column lookup by name if '@id' not matching DataFrame
        col_name = numeric_fields[0].get('name', numeric_field_id)
        # Use column name for DataFrame
        col_for_df = col_name if col_name in df.columns else numeric_field_id

        # Filter
        threshold = 10
        filtered_df = df[df[col_for_df] > threshold]
        print(f"Filtered records with {col_for_df} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{col_for_df}_normalized"] = (filtered_df[col_for_df] - filtered_df[col_for_df].mean()) / filtered_df[col_for_df].std()
        print(f"Normalized {col_for_df} for filtered records:")
        display(filtered_df[[col_for_df, f"{col_for_df}_normalized"].head())

        # Group by another field, e.g., 'Adrenal_CT' if present
        group_candidate_fields = [f for f in record_set.get('field', []) if f.get('dataType', '') == 'Text']
        if group_candidate_fields:
            group_field_id = group_candidate_fields[0]['@id']
            group_col_name = group_candidate_fields[0].get('name', group_field_id)
            group_col_for_df = group_col_name if group_col_name in df.columns else group_field_id
            if group_col_for_df in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_col_for_df)[col_for_df].mean().reset_index()
                print(f"Grouped data by {group_col_for_df}:")
                display(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between relevant fields.

In [ ]:
# Visualize distribution of the numeric field from Table 2 (hormone levels)
if len(record_set_ids) > 1 and numeric_fields:
    df = dataframes[table2_id]
    col_for_df = numeric_fields[0].get('name', numeric_fields[0]['@id'])
    if col_for_df in df.columns:
        plt.figure(figsize=(8,5))
        plt.hist(df[col_for_df].dropna(), bins=10, color='skyblue', edgecolor='black')
        plt.xlabel(col_for_df)
        plt.ylabel('Count')
        plt.title(f'Distribution of {col_for_df} in Table 2')
        plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR² dataset package using its Croissant schema, reviewed the record sets and fields (by `@id`), extracted the data into DataFrames, and performed basic exploratory analyses including filtering, normalization, grouping, and visualization. 

The dataset demonstrates genotype–phenotype heterogeneity for NC-21OHD-associated infertility, but is limited by its small size and specific cohort. For additional analyses, you may review different record sets or fields, always referencing them by their `@id`.